# 4 — Feature Engineering for Sensor Models

**Sensor Intelligence Platform** — analytical walkthrough (4 / 7)

Tabular models do not see a time series; they see a **feature matrix**. `FeatureBuilder` turns the tidy long-format frame into one — rolling statistics, lags, cyclical calendar encodings, and sensor-health/missingness features — computing every windowed feature *within* each sensor group so information never leaks across channels.

1. Build the feature matrix and inspect the feature families.
2. See why per-sensor grouping prevents cross-channel leakage.
3. Visualise rolling, calendar, and health features.
4. Measure which features actually drive a forecaster.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

NAVY, ORANGE, TEAL, RED, GREY = "#1f3a5f", "#e8893b", "#2a9d8f", "#c0392b", "#9aa0ad"

## 4.1 A small labelled window

We simulate four channels of the reference fleet for six days and inject a short temperature spike, so the feature matrix has both clean structure and a fault to describe.

In [2]:
from sensor_intelligence.simulation import (
    SensorSimulator, SimulationConfig, AnomalyInjection, default_fleet,
)

PERIOD = 96
channels = ['temperature', 'vibration', 'flow_rate', 'motor_current']
fleet = [s for s in default_fleet() if s.sensor_id in channels]
config = SimulationConfig(
    sensors=fleet, n_steps=PERIOD * 6, step_seconds=900, seed=17,
    anomalies=[AnomalyInjection('temperature', start_step=PERIOD * 3 + 20,
                                duration=4, magnitude=16.0)],
)
df = SensorSimulator(config).run()
print(df.shape, '->', sorted(df.sensor_id.unique()))
df.head(3)

(2304, 5) -> ['flow_rate', 'motor_current', 'temperature', 'vibration']


,sensor_id,timestamp,value,unit,is_anomaly
0,flow_rate,2024-01-01 00:00:00,114.397639,m3/h,False
1,flow_rate,2024-01-01 00:15:00,122.991065,m3/h,False
2,flow_rate,2024-01-01 00:30:00,123.656300,m3/h,False


## 4.2 The feature families

`FeatureBuilder().transform` adds four families in one call. The defaults produce rolling mean/std/min/max over three windows, three lags, calendar fields with sin/cos encodings, and sensor-health features (sampling gap, missingness, observed count).

In [3]:
from sensor_intelligence.features import FeatureBuilder

feat = FeatureBuilder().transform(df)
base = {'sensor_id', 'timestamp', 'value', 'unit', 'is_anomaly'}
families = {
    'rolling': [c for c in feat if c.startswith('roll_')],
    'lag': [c for c in feat if c.startswith('lag_')],
    'calendar': ['hour', 'dayofweek', 'is_weekend', 'month',
                 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'],
    'health': ['gap_seconds', 'is_missing', 'missing_rate', 'observed_count'],
}
for name, cols in families.items():
    print(f'{name:9s} ({len(cols):2d}): {", ".join(cols)}')
print(f'\n{feat.shape[1]} columns total, {feat.shape[0]} rows')
feat[['sensor_id', 'timestamp', 'value', 'roll_mean_15', 'lag_1', 'hour_sin']].head(3)

rolling   (12): roll_mean_5, roll_std_5, roll_min_5, roll_max_5, roll_mean_15, roll_std_15, roll_min_15, roll_max_15, roll_mean_60, roll_std_60, roll_min_60, roll_max_60
lag       ( 3): lag_1, lag_2, lag_3
calendar  ( 8): hour, dayofweek, is_weekend, month, hour_sin, hour_cos, dow_sin, dow_cos
health    ( 4): gap_seconds, is_missing, missing_rate, observed_count

32 columns total, 2304 rows


,sensor_id,timestamp,value,roll_mean_15,lag_1,hour_sin
0,flow_rate,2024-01-01 00:00:00,114.397639,114.397639,NaN,0.000000
1,flow_rate,2024-01-01 00:15:00,122.991065,118.694352,114.397639,0.065403
2,flow_rate,2024-01-01 00:30:00,123.656300,120.348335,122.991065,0.130526


## 4.3 Leakage safety: features stop at the sensor boundary

Windowed and lag features are computed per sensor group. At the **first row of each sensor** the lags are therefore undefined (`NaN`) rather than borrowed from the previous sensor in the frame — direct proof that no information crosses the channel boundary.

In [4]:
firsts = feat.groupby('sensor_id', sort=False).head(1)
firsts[['sensor_id', 'lag_1', 'lag_2', 'lag_3', 'roll_mean_5']]

,sensor_id,lag_1,lag_2,lag_3,roll_mean_5
0,flow_rate,NaN,NaN,NaN,114.397639
576,motor_current,NaN,NaN,NaN,13.699229
1152,temperature,NaN,NaN,NaN,62.660757
1728,vibration,NaN,NaN,NaN,0.391350


## 4.4 Rolling statistics track level and volatility

The rolling mean is a smoothed version of the signal; the rolling std is a local volatility estimate that **spikes around the injected fault** — the same signal a residual anomaly detector keys on.

In [5]:
t = feat[feat.sensor_id == 'temperature']
fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a1.plot(t.timestamp, t.value, color=GREY, lw=0.8, label='value')
a1.plot(t.timestamp, t.roll_mean_15, color=NAVY, lw=1.6, label='roll_mean_15')
a1.fill_between(t.timestamp, t.roll_mean_15 - t.roll_std_15,
                t.roll_mean_15 + t.roll_std_15, color=NAVY, alpha=0.15,
                label='±roll_std_15')
faults = t[t.is_anomaly]
a1.scatter(faults.timestamp, faults.value, color=RED, s=18, zorder=5, label='fault')
a1.set(title='Temperature: rolling mean and volatility band', ylabel='C')
a1.legend(loc='upper left', fontsize=9, ncol=2)
a2.plot(t.timestamp, t.roll_std_15, color=ORANGE, lw=1.4)
a2.set(title='Rolling std spikes at the fault', ylabel='roll_std_15 (C)', xlabel='time')
fig.autofmt_xdate(); fig.tight_layout()

## 4.5 Calendar encodings make time cyclical

Encoding the hour as a raw integer puts 23:00 and 00:00 maximally far apart. The sin/cos pair maps the clock onto a circle so midnight and 1 a.m. are neighbours — a representation a linear or tree model can use directly.

In [6]:
day = (feat[feat.sensor_id == 'temperature']
       .drop_duplicates('hour').sort_values('hour'))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
sc = a1.scatter(day.hour_sin, day.hour_cos, c=day.hour, cmap='twilight', s=90)
for _, r in day.iterrows():
    a1.annotate(int(r.hour), (r.hour_sin, r.hour_cos), fontsize=7,
                ha='center', va='center')
a1.set(title='Hour mapped onto a circle', xlabel='hour_sin', ylabel='hour_cos')
a1.set_aspect('equal')
one = feat[feat.sensor_id == 'flow_rate'].head(PERIOD)
a2.plot(one.timestamp, one.hour_sin, color=NAVY, label='hour_sin')
a2.plot(one.timestamp, one.hour_cos, color=ORANGE, label='hour_cos')
a2.set(title='Smooth daily encodings', xlabel='time'); a2.legend(fontsize=9)
fig.colorbar(sc, ax=a1, shrink=0.8, label='hour')
fig.autofmt_xdate(); fig.tight_layout()

## 4.6 Health features expose data-quality gaps

Real feeds drop samples. We blank out a six-hour stretch of vibration readings and watch the health features react: `is_missing` flips and the rolling `missing_rate` climbs. An operator can alert on these before a model ever runs.

In [7]:
dirty = df.copy()
t0 = dirty.timestamp.min()
mask = ((dirty.sensor_id == 'vibration')
        & (dirty.timestamp >= t0 + pd.Timedelta(days=2))
        & (dirty.timestamp < t0 + pd.Timedelta(days=2, hours=6)))
dirty.loc[mask, 'value'] = np.nan
hfeat = FeatureBuilder().transform(dirty)
v = hfeat[hfeat.sensor_id == 'vibration']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(v.timestamp, v.missing_rate, color=RED, lw=1.6, label='missing_rate (rolling)')
ax.fill_between(v.timestamp, 0, v.is_missing, color=GREY, alpha=0.3, step='mid',
                label='is_missing')
ax.set(title='Sensor-health features during a 6-hour dropout',
       xlabel='time', ylabel='rate'); ax.legend(loc='upper right', fontsize=9)
fig.autofmt_xdate(); fig.tight_layout()
print(f'dropped {int(v.is_missing.sum())} samples; '
      f'peak missing_rate = {v.missing_rate.max():.2f}')

dropped 24 samples; peak missing_rate = 1.00


## 4.7 Which features drive a forecast?

The payoff: we train a gradient-boosted regressor to predict the **next** temperature value from the engineered features and rank them by **permutation importance** — how much validation error grows when each feature is shuffled. Recent lags and the rolling mean dominate, with the calendar terms supplying the daily structure.

In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

tf = feat[feat.sensor_id == 'temperature'].copy()
tf['target'] = tf.value.shift(-1)
model_cols = [c for c in feat.columns if c not in base]
data = tf.dropna(subset=model_cols + ['target'])
X, y = data[model_cols], data['target']
split = int(len(X) * 0.7)
Xtr, Xte, ytr, yte = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]

reg = HistGradientBoostingRegressor(max_depth=3, max_iter=200,
                                    random_state=0).fit(Xtr, ytr)
imp = permutation_importance(reg, Xte, yte, n_repeats=10, random_state=0)
ranked = (pd.Series(imp.importances_mean, index=model_cols)
            .sort_values(ascending=False).head(12))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranked.index[::-1], ranked.values[::-1], color=NAVY)
ax.set(title='Permutation importance for next-step temperature',
       xlabel='increase in error when shuffled')
fig.tight_layout()
print(f'validation MAE: {np.mean(np.abs(reg.predict(Xte) - yte)):.3f} C')

validation MAE: 0.992 C


## Takeaways

- `FeatureBuilder` produces four feature families from a tidy frame in one call, all **leakage-safe per sensor**.
- **Rolling** features track level and volatility; **calendar** sin/cos encodings make time cyclical; **health** features expose dropouts before modelling.
- Permutation importance confirms recent **lags and the rolling mean** carry most of the signal, with calendar terms adding daily structure.
- These are exactly the features `TabularForecaster` builds internally — see it forecast in [notebook 2](02_forecasting_and_backtesting.ipynb).